## Grupp4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer

# Ladda data
df = pd.read_csv('historical_data.csv')


EDA

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
target_counts = df['is_suspicious'].value_counts(normalize=True)
print(f"Fördelning:\n{target_counts}")

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='is_suspicious', data=df)
plt.title('Fördelning av Misstänkta vs Normala händelser')
plt.show()

Vårt dataset är obalanserat.
Om vi gissar att alla är hederliga har vi 90 % rätt, men vi stoppar 0 % av bedrägerierna.
Vi måste därför prioritera Precision (för att inte flagga oskyldiga i onödan).

In [ ]:
missing_data = df.isnull().sum().sort_values(ascending=False)
print("\n--- Antal saknade värden per kolumn ---")
print(missing_data[missing_data > 0])


In [ ]:
print(df["region"].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='account_age_days', hue='is_suspicious', common_norm=False, fill=True)
plt.title('Distribution av kontoålder: Normala vs Misstänkta')
plt.xlabel('Dagar sedan kontot skapades')
plt.xlim(0, 500)
plt.show()

Vi ser en tydlig 'peak' för misstänkta händelser bland nyskapade konton (under 30 dagar). Detta kan var en kritisk signal för modellen. Äldre konton är generellt sett mycket säkrare.

## Pipeline

In [ ]:
target_name = "is_suspicious"
X = df.drop(columns=[target_name,"id"])
y = df[target_name]

# make numeric/category feature
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object","string"]).columns.tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("numeric_features :", numeric_features)
print("\ncategorical_features :", categorical_features)

## Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)

    ],
    remainder="drop"
)


### Modeller 

### Baslinjemodell (Baseline)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),    
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}

dummy_model = DummyClassifier(strategy="constant", constant=1, random_state=42)
dummy_model.fit(X_train, y_train)


dummy_scores = cross_validate(dummy_model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)

dummy_df = pd.DataFrame(dummy_scores)
print("Dummy Model Scores:")
print(dummy_df[["test_accuracy", "test_precision", "test_recall", "test_f1"]].mean())


dummy_model2 = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_model2.fit(X_train, y_train)


dummy_scores2 = cross_validate(dummy_model2, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)

dummy_df2 = pd.DataFrame(dummy_scores2)
print("Dummy Model Scores:")
print(dummy_df2[["test_accuracy", "test_precision", "test_recall", "test_f1"]].mean())

In [ ]:
logistic= Pipeline([
    ("preprocessor", preprocess),
    ("classifier", LogisticRegression(max_iter=2500, random_state=42))
])


random_forest = Pipeline([
    ("preprocessor", preprocess),
    ("classifier", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42))
])


decision_tree = Pipeline([
    ("preprocessor", preprocess),
    ("classifier", DecisionTreeClassifier(random_state=42))
])


In [ ]:
models = {
    "Logistic Regression": logistic,
    "Random Forest": random_forest,
    "Decision Tree": decision_tree
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained:", name)


### Krosvalidering (Cross-validation)
Vårt primära mätvärde för affärsnyttan är Precision. Vi utvärderar dock även modellerna utifrån andra mätetal för att säkerställa att de är stabila överlag.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score),    
    "recall": make_scorer(recall_score),
    "f1": make_scorer(f1_score)
}

cv_results = {}
for name, model in models.items():
    cv_results[name] = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False)


cv_summary = pd.DataFrame({
    name: {metric: result[f'test_{metric}'].mean() for metric in scoring.keys()}
    for name, result in cv_results.items()
}).T

print(cv_summary)


### Den bästa modellen 
Baserat på resultaten från korsvalideringen presterar Logistisk Regression bäst och väljs därmed som vår huvudmodell.

In [ ]:
best_model = models["Logistic Regression"]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Precision:", precision_score(y_test, y_pred))
print("Test Recall:", recall_score(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))


***Från matematisk prestation till affärsnytta:***

Även om Logistisk Regression presterar bäst av våra testade algoritmer, visar mätetalen (som F1-score och Recall) att modellen fortfarande gör misstag. I en verklig kontext innebär varje misstag att antingen en bedragare slipper undan, eller att en oskyldig användare får sitt konto spärrat.

Eftersom kundtjänst (Lina) har ett tydligt kravkort: "Fokus är beslut utifrån kravkortet – oskyldiga ska inte bli arga", kan vi inte nöja oss med standardgränsvärdet på 50 %. Vi måste nu kalibrera modellen utifrån affärens smärtgräns.

### Utvärdering av olika tröskelvärden (Thresholds)

In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]
print("Antal sannolikheter för testdata:", len(test_proba))

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score

thresholds = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]

print("Threshold | Totalt Flaggade | Oskyldiga (False Positives) | Precision (Hur säkra vi är)")
print("-" * 85)

for t in thresholds:
    y_pred_custom = (test_proba >= t).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
    
    total_flagged = tp + fp
    precision = precision_score(y_test, y_pred_custom, zero_division=0)
    
    print(f"{t:<9} | {total_flagged:<15} | {fp:<29} | {precision:.1%}")

**Analys av tröskelvärden:**

Som vi ser i tabellen ovan är standardvärdet 0.5 (50%) problematiskt för Lina. Vid 0.5 flaggar vi 23 personer, men 11 av dem är helt oskyldiga (False Positives). Det är en felmarginal på nästan 50%, vilket skulle skapa ett enormt tryck på kundtjänst.

Notera även att vid 0.8 och uppåt flaggas 0 personer. Det betyder att modellens högsta sannolikhet för testdatat ligger någonstans i 70-procentsspannet. Vi kan alltså inte bara "höja ribban" till 90% för att vara säkra, då skulle vi inte fånga några skurkar alls.

### Alternativ 1: Finjustering av tröskelvärde

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score

micro_thresholds = np.arange(0.61, 0.70, 0.01)

print("Threshold | Totalt Flaggade | Oskyldiga (FP) | Precision")
print("-" * 65)

for t in micro_thresholds:
    y_pred_custom = (test_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
    
    total_flagged = tp + fp
    precision = precision_score(y_test, y_pred_custom, zero_division=0)
    
    print(f"{t:<9.2f} | {total_flagged:<15} | {fp:<14} | {precision:.1%}")

**Strategi 1: Optimering av tröskelvärde**

Genom att titta närmare på spannet 0.60–0.69 ser vi hur precisionen förändras för varje procentenhet.

Slutsats: Genom att ligga runt 0.65 kan vi halvera antalet oskyldiga som drabbas jämfört med standardinställningen, samtidigt som vi behåller en rimlig volym för granskning. Detta balanserar Linas krav på att "oskyldiga inte ska bli arga".

### Alternativ 2: Prioriteringsregel (Topp 10)


In [ ]:
results_df = pd.DataFrame({'Target': y_test, 'Probability': test_proba})

results_df = results_df.sort_values(by='Probability', ascending=False)

# Välj ut top 10
top_x = 10
top_x_df = results_df.head(top_x)

fp_in_top_x = len(top_x_df[(top_x_df['Target'] == 0)])
tp_in_top_x = len(top_x_df[(top_x_df['Target'] == 1)])
precision_top_x = tp_in_top_x / top_x if top_x > 0 else 0

print(f"Bland de top {top_x} mest misstänkta fallen:")
print(f"Sanna skurkar (TP): {tp_in_top_x}")
print(f"Oskyldiga (False Positives): {fp_in_top_x}")
print(f"Precision: {precision_top_x:.1%}")

**Strategi 2: Topp-lista för granskning**

Istället för ett fast gränsvärde kan vi leverera en daglig "Topp 10"-lista till Lina.

Fördel: Detta ger kundtjänst en förutsägbar arbetsbörda. Vi fokuserar resurserna där sannolikheten för bedrägeri är som högst. Som visas i resultatet ovan är precisionen högre i den absoluta toppen av listan.

### Exempel på en felaktigt flaggad användare (False Positive)

In [ ]:
# Sätt gränsvärdet vi vill undersöka (Vi använder 0,60 som tröskelvärde i detta exempel eftersom vi vet att det flaggar 5 oskyldiga personer)
vald_threshold = 0.60

oskyldiga_mask = (y_test == 0)

# Hitta de som modellen trodde var skurkar (Sannolikhet >= 0.60)
flaggade_mask = (test_proba >= vald_threshold)

# Plocka ut dessa rader från testdatan
falska_positiva = X_test[oskyldiga_mask & flaggade_mask]

print(f"Antal oskyldiga användare som flaggades vid gränsvärdet {vald_threshold}: {len(falska_positiva)}")
print("-" * 80)
print("Här är ett konkret exempel på en oskyldig användare som modellen flaggade av misstag:")

# Visa den allra första oskyldiga användaren i en tabell
display(falska_positiva.head(1))

**Varför flaggas oskyldiga? (Case-studie för Lina/Customer Supprt)**

Här har vi extraherat ett konkret exempel på en användare som de facto är oskyldig (Target=0), men som modellen ändå bedömde som en hög risk (Sannolikhet > 0.60).

Om vi tittar på just denna användares data förstår vi exakt varför algoritmen reagerade:

- Nytt och overifierat konto: Kontot är bara drygt två veckor gammalt (account_age_days: 17) och saknar helt verifiering (verification_level: 0).

- Regelbrott i chatten: Användaren har försökt styra konversationen bort från vår plattform (contains_off_platform: 1), till exempel genom att be om att få ta det via SMS eller WhatsApp. Detta är ett av de vanligaste beteendena hos riktiga bedragare.

- Tidigare misstankar: Användaren har redan blivit anmäld av någon annan den senaste månaden (prev_reports_30d: 1).

**Slutsats för Kundtjänst: >** Modellen gör egentligen "rätt" som reagerar – datan skriker bedragare. Men eftersom facit visar att personen är oskyldig, bevisar detta att legitima användare ibland agerar klumpigt eller bryter mot plattformens riktlinjer utan att ha ett brottsligt uppsåt. Om vi tillämpar en policy med automatisk avstängning (ban) här, kommer vi att få en mycket arg kund som ringer till Lina och anser sig vara orättvist behandlad.

**Rekommenderad Policy ("Mjuk friktion"): >** Istället för permanent avstängning föreslår vi att systemet fångar upp dessa beteenden och lägger in en tvingande spärr – till exempel att annonsen pausas och användaren omedelbart måste verifiera sig med BankID/SMS (verification_level måste öka). Detta stoppar automatiserade skurkar direkt, medan klumpiga men oskyldiga användare kan låsa upp sina konton själva utan att belasta kundtjänst.